# AW-MAE Football 


In [19]:
from __future__ import annotations

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from collections import defaultdict, deque
from typing import Tuple, Optional

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor

pd.set_option("display.max_columns", 200)

SEED = 42
HOME_ELO_ADV = 55.0
ELO_K = 20.0
MAX_GOALS = 10
FAST_MODE = False

DIRECT_ITERS = 450 if FAST_MODE else 950
STRUCT_ITERS = 350 if FAST_MODE else 750
N_FOLDS = 5
FOLD_START_QUANTILE = 0.70


## Load data dan metric


In [20]:
TRAIN_PATH = "./dataset/train.csv"
TEST_PATH = "./dataset/test.csv"
SUB_PATH = "submission.csv"  
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
train["date"] = pd.to_datetime(train["date"])
test["date"] = pd.to_datetime(test["date"])

sub = pd.DataFrame({"Id": test["Id"], "team_goals": 0, "opp_goals": 0})

print("TRAIN:", train.shape, train["date"].min().date(), "->", train["date"].max().date())
print("TEST :", test.shape, test["date"].min().date(), "->", test["date"].max().date())

def get_tournament_weight(tournament: str) -> float:
    """
    Mapping bobot turnamen berdasarkan info terbaru dari screenshot.
    Kalau nanti di halaman kompetisi ada mapping yang lebih detail,
    tinggal tambahkan keyword di sini.
    """
    t = str(tournament).lower().strip()

    if "fifa world cup" in t or t == "world cup":
        return 2.00

    if "afc championship" in t or "afc asian cup" in t or "asian cup" in t:
        return 1.80

    if "friendly" in t:
        return 0.96

    return 1.20


EXACT_PENALTY = 0.30
OUTCOME_PENALTY = 0.25
GD_PENALTY = 0.15
WRONG_OUTCOME_MULTIPLIER = 1.50
NONLINEAR_POWER = 1.50


def _outcome(a: int, b: int) -> int:
    if a > b:
        return 1
    if a < b:
        return -1
    return 0


def official_match_loss(
    y_team_true,
    y_opp_true,
    y_team_pred,
    y_opp_pred,
):
    """
    Menghasilkan loss per match sesuai definisi metric terbaru:
    1. base MAE
    2. tambah penalti jika exact / outcome / GD salah
    3. kalau outcome salah, total error dikali 1.5
    4. pangkatkan 1.5
    """
    y_team_true = np.asarray(y_team_true).astype(int)
    y_opp_true = np.asarray(y_opp_true).astype(int)
    y_team_pred = np.asarray(y_team_pred).astype(int)
    y_opp_pred = np.asarray(y_opp_pred).astype(int)

    base_mae = (
        np.abs(y_team_true - y_team_pred) +
        np.abs(y_opp_true - y_opp_pred)
    ) / 2.0

    exact_hit = (y_team_true == y_team_pred) & (y_opp_true == y_opp_pred)

    true_outcome = np.vectorize(_outcome)(y_team_true, y_opp_true)
    pred_outcome = np.vectorize(_outcome)(y_team_pred, y_opp_pred)
    outcome_hit = (true_outcome == pred_outcome)

    true_gd = y_team_true - y_opp_true
    pred_gd = y_team_pred - y_opp_pred
    gd_hit = (true_gd == pred_gd)

    # Penalti ditambahkan jika salah pada aspek tersebut
    penalty = (
        (~exact_hit).astype(float) * EXACT_PENALTY +
        (~outcome_hit).astype(float) * OUTCOME_PENALTY +
        (~gd_hit).astype(float) * GD_PENALTY
    )

    raw_error = base_mae + penalty

    raw_error = np.where(
        outcome_hit,
        raw_error,
        raw_error * WRONG_OUTCOME_MULTIPLIER
    )

    final_loss = raw_error ** NONLINEAR_POWER
    return final_loss


def awmae_score(
    y_team_true,
    y_opp_true,
    y_team_pred,
    y_opp_pred,
    weights=None,
) -> float:
    losses = official_match_loss(
        y_team_true=y_team_true,
        y_opp_true=y_opp_true,
        y_team_pred=y_team_pred,
        y_opp_pred=y_opp_pred,
    )

    if weights is None:
        weights = np.ones(len(losses), dtype=float)
    else:
        weights = np.asarray(weights, dtype=float)

    return np.average(losses, weights=weights)

train["tournament_weight"] = train["tournament"].apply(get_tournament_weight)
test["tournament_weight"] = test["tournament"].apply(get_tournament_weight)


TRAIN: (78772, 47) 1872-11-30 -> 2011-08-04
TEST : (42422, 20) 2011-08-06 -> 2026-03-31


## State engine dan causal feature builder


In [21]:
def clean_num(x):
    if pd.isna(x):
        return np.nan
    try:
        x = float(x)
    except Exception:
        return np.nan
    return np.nan if x == -9999 else x

def safe_log1p(x):
    if pd.isna(x) or x < 0:
        return np.nan
    return np.log1p(x)

def mean_or_nan(values):
    values = list(values)
    return np.nan if len(values) == 0 else float(np.mean(values))

def sum_or_nan(values):
    values = list(values)
    return np.nan if len(values) == 0 else float(np.sum(values))

def result_points(gf: float, ga: float) -> int:
    if gf > ga:
        return 3
    if gf == ga:
        return 1
    return 0

def expected_score(ra: float, rb: float) -> float:
    return 1.0 / (1.0 + 10 ** ((rb - ra) / 400.0))

def new_team_state():
    return {
        "points5": deque(maxlen=5), "points10": deque(maxlen=10),
        "gd5": deque(maxlen=5),
        "gf5": deque(maxlen=5), "ga5": deque(maxlen=5),
        "gf10": deque(maxlen=10), "ga10": deque(maxlen=10),
        "win10": deque(maxlen=10), "draw10": deque(maxlen=10),
        "home_points5": deque(maxlen=5), "away_points5": deque(maxlen=5),
        "home_gf5": deque(maxlen=5), "away_gf5": deque(maxlen=5),
        "last_date": pd.NaT, "matches": 0, "elo": 1500.0
    }

def new_h2h_state():
    return {
        "points5": deque(maxlen=5), "gd5": deque(maxlen=5),
        "gf5": deque(maxlen=5), "ga5": deque(maxlen=5),
        "matches": 0
    }

def make_reverse_row(row: pd.Series) -> pd.Series:
    rev = row.copy()
    rev["Id"] = f"{row['match_id']}_{row['opponent']}"
    rev["team"] = row["opponent"]
    rev["opponent"] = row["team"]
    if "team_goals" in row.index:
        rev["team_goals"] = row["opp_goals"]
        rev["opp_goals"] = row["team_goals"]
    rev["is_home"] = 0 if int(row.get("is_home", 0) or 0) == 1 else 1
    for a, b in [
        ("confederation_team", "confederation_opp"),
        ("population_team", "population_opp"),
        ("gdp_per_capita_team", "gdp_per_capita_opp"),
        ("distance_travel_team", "distance_travel_opp"),
    ]:
        if a in row.index and b in row.index:
            rev[a], rev[b] = row.get(b), row.get(a)
    return rev

def build_feature_row(row: pd.Series, team_state: dict, opp_state: dict, h2h_state: dict) -> dict:
    date = pd.Timestamp(row["date"])
    pop_team = clean_num(row.get("population_team"))
    pop_opp = clean_num(row.get("population_opp"))
    gdp_team = clean_num(row.get("gdp_per_capita_team"))
    gdp_opp = clean_num(row.get("gdp_per_capita_opp"))
    altitude = clean_num(row.get("altitude_venue"))
    dist_team = clean_num(row.get("distance_travel_team"))
    dist_opp = clean_num(row.get("distance_travel_opp"))
    temp = clean_num(row.get("temperature_venue"))

    rest_team = np.nan if pd.isna(team_state["last_date"]) else (date - pd.Timestamp(team_state["last_date"])).days
    rest_opp = np.nan if pd.isna(opp_state["last_date"]) else (date - pd.Timestamp(opp_state["last_date"])).days

    f = {
        "Id": row["Id"], "match_id": row["match_id"], "date": date,
        "team": str(row.get("team", "__NA__")), "opponent": str(row.get("opponent", "__NA__")),
        "gender": str(row.get("gender", "__NA__")), "tournament": str(row.get("tournament", "__NA__")),
        "venue_country": str(row.get("venue_country", "__NA__")),
        "confederation_team": str(row.get("confederation_team", "__NA__")),
        "confederation_opp": str(row.get("confederation_opp", "__NA__")),
        "is_home": int(row.get("is_home", 0) or 0), "neutral": int(row.get("neutral", 0) or 0),
        "tournament_weight": float(row.get("tournament_weight", 1.0)),
        "year": date.year, "month": date.month, "dayofyear": date.dayofyear, "dayofweek": date.dayofweek,
        "month_sin": np.sin(2*np.pi*date.month/12.0), "month_cos": np.cos(2*np.pi*date.month/12.0),
        "doy_sin": np.sin(2*np.pi*date.dayofyear/366.0), "doy_cos": np.cos(2*np.pi*date.dayofyear/366.0),
        "same_conf": int(str(row.get("confederation_team", "")) == str(row.get("confederation_opp", ""))),
        "team_matches_played": team_state["matches"], "opp_matches_played": opp_state["matches"],
        "rest_days_team": rest_team, "rest_days_opp": rest_opp,
        "team_points_last5": sum_or_nan(team_state["points5"]), "opp_points_last5": sum_or_nan(opp_state["points5"]),
        "team_points_last10": sum_or_nan(team_state["points10"]), "opp_points_last10": sum_or_nan(opp_state["points10"]),
        "team_gd_last5": sum_or_nan(team_state["gd5"]), "opp_gd_last5": sum_or_nan(opp_state["gd5"]),
        "team_avg_goals_last5": mean_or_nan(team_state["gf5"]), "opp_avg_goals_last5": mean_or_nan(opp_state["gf5"]),
        "team_avg_conceded_last5": mean_or_nan(team_state["ga5"]), "opp_avg_conceded_last5": mean_or_nan(opp_state["ga5"]),
        "team_avg_goals_last10": mean_or_nan(team_state["gf10"]), "opp_avg_goals_last10": mean_or_nan(opp_state["gf10"]),
        "team_avg_conceded_last10": mean_or_nan(team_state["ga10"]), "opp_avg_conceded_last10": mean_or_nan(opp_state["ga10"]),
        "team_win_rate_last10": mean_or_nan(team_state["win10"]), "opp_win_rate_last10": mean_or_nan(opp_state["win10"]),
        "team_draw_rate_last10": mean_or_nan(team_state["draw10"]), "opp_draw_rate_last10": mean_or_nan(opp_state["draw10"]),
        "team_home_points_last5": sum_or_nan(team_state["home_points5"]), "opp_away_points_last5": sum_or_nan(opp_state["away_points5"]),
        "team_home_gf_last5": mean_or_nan(team_state["home_gf5"]), "opp_away_gf_last5": mean_or_nan(opp_state["away_gf5"]),
        "elo_team": team_state["elo"], "elo_opp": opp_state["elo"], "elo_diff": team_state["elo"] - opp_state["elo"],
        "h2h_points_last5": sum_or_nan(h2h_state["points5"]), "h2h_gd_last5": sum_or_nan(h2h_state["gd5"]),
        "h2h_avg_goals_last5": mean_or_nan(h2h_state["gf5"]), "h2h_avg_conceded_last5": mean_or_nan(h2h_state["ga5"]),
        "h2h_matches": h2h_state["matches"],
        "population_team_log": safe_log1p(pop_team), "population_opp_log": safe_log1p(pop_opp),
        "gdp_team_log": safe_log1p(gdp_team), "gdp_opp_log": safe_log1p(gdp_opp),
        "altitude_venue_clean": altitude, "distance_travel_team_clean": dist_team,
        "distance_travel_opp_clean": dist_opp, "temperature_venue_clean": temp,
    }

    f["matches_played_diff"] = f["team_matches_played"] - f["opp_matches_played"]
    f["rest_days_diff"] = (rest_team if not pd.isna(rest_team) else 0) - (rest_opp if not pd.isna(rest_opp) else 0)
    f["points_last5_diff"] = (f["team_points_last5"] if not pd.isna(f["team_points_last5"]) else 0) - (f["opp_points_last5"] if not pd.isna(f["opp_points_last5"]) else 0)
    f["points_last10_diff"] = (f["team_points_last10"] if not pd.isna(f["team_points_last10"]) else 0) - (f["opp_points_last10"] if not pd.isna(f["opp_points_last10"]) else 0)
    f["gd_last5_diff"] = (f["team_gd_last5"] if not pd.isna(f["team_gd_last5"]) else 0) - (f["opp_gd_last5"] if not pd.isna(f["opp_gd_last5"]) else 0)
    f["avg_goals_last5_diff"] = (f["team_avg_goals_last5"] if not pd.isna(f["team_avg_goals_last5"]) else 0) - (f["opp_avg_goals_last5"] if not pd.isna(f["opp_avg_goals_last5"]) else 0)
    f["avg_conceded_last5_diff"] = (f["team_avg_conceded_last5"] if not pd.isna(f["team_avg_conceded_last5"]) else 0) - (f["opp_avg_conceded_last5"] if not pd.isna(f["opp_avg_conceded_last5"]) else 0)
    f["win_rate_last10_diff"] = (f["team_win_rate_last10"] if not pd.isna(f["team_win_rate_last10"]) else 0) - (f["opp_win_rate_last10"] if not pd.isna(f["opp_win_rate_last10"]) else 0)
    f["draw_rate_last10_diff"] = (f["team_draw_rate_last10"] if not pd.isna(f["team_draw_rate_last10"]) else 0) - (f["opp_draw_rate_last10"] if not pd.isna(f["opp_draw_rate_last10"]) else 0)
    f["pop_log_diff"] = (f["population_team_log"] if not pd.isna(f["population_team_log"]) else 0) - (f["population_opp_log"] if not pd.isna(f["population_opp_log"]) else 0)
    f["gdp_log_diff"] = (f["gdp_team_log"] if not pd.isna(f["gdp_team_log"]) else 0) - (f["gdp_opp_log"] if not pd.isna(f["gdp_opp_log"]) else 0)
    f["travel_diff"] = (dist_team if not pd.isna(dist_team) else 0) - (dist_opp if not pd.isna(dist_opp) else 0)
    f["altitude_missing"] = int(pd.isna(altitude))
    f["temp_missing"] = int(pd.isna(temp))
    return f

def update_match_states(row_a: pd.Series, row_b: pd.Series, goals_a: int, goals_b: int, team_states: dict, h2h_states: dict):
    team_a = str(row_a["team"]); team_b = str(row_a["opponent"]); date = pd.Timestamp(row_a["date"])
    state_a = team_states[team_a]; state_b = team_states[team_b]
    is_home_a = int(row_a.get("is_home", 0) or 0)
    is_home_b = int(row_b.get("is_home", 0) or 0) if row_b is not None else (0 if is_home_a == 1 else 1)
    neutral = int(row_a.get("neutral", 0) or 0)

    pts_a = result_points(goals_a, goals_b); pts_b = result_points(goals_b, goals_a)

    for st, gf, ga, pts, home_flag in [
        (state_a, goals_a, goals_b, pts_a, is_home_a),
        (state_b, goals_b, goals_a, pts_b, is_home_b),
    ]:
        st["points5"].append(pts); st["points10"].append(pts); st["gd5"].append(gf-ga)
        st["gf5"].append(gf); st["ga5"].append(ga); st["gf10"].append(gf); st["ga10"].append(ga)
        st["win10"].append(int(gf > ga)); st["draw10"].append(int(gf == ga))
        if home_flag == 1:
            st["home_points5"].append(pts); st["home_gf5"].append(gf)
        else:
            st["away_points5"].append(pts); st["away_gf5"].append(gf)
        st["last_date"] = date; st["matches"] += 1

    h2h_states[(team_a, team_b)]["points5"].append(pts_a)
    h2h_states[(team_a, team_b)]["gd5"].append(goals_a-goals_b)
    h2h_states[(team_a, team_b)]["gf5"].append(goals_a)
    h2h_states[(team_a, team_b)]["ga5"].append(goals_b)
    h2h_states[(team_a, team_b)]["matches"] += 1

    h2h_states[(team_b, team_a)]["points5"].append(pts_b)
    h2h_states[(team_b, team_a)]["gd5"].append(goals_b-goals_a)
    h2h_states[(team_b, team_a)]["gf5"].append(goals_b)
    h2h_states[(team_b, team_a)]["ga5"].append(goals_a)
    h2h_states[(team_b, team_a)]["matches"] += 1

    adv_a = HOME_ELO_ADV if (neutral == 0 and is_home_a == 1) else 0.0
    adv_b = HOME_ELO_ADV if (neutral == 0 and is_home_b == 1) else 0.0
    ra = state_a["elo"] + adv_a; rb = state_b["elo"] + adv_b
    ea = expected_score(ra, rb)
    sa = 1.0 if goals_a > goals_b else 0.5 if goals_a == goals_b else 0.0
    delta = ELO_K * (sa - ea)
    state_a["elo"] += delta; state_b["elo"] -= delta

def prepare_match_groups(df: pd.DataFrame):
    df = df.copy().sort_values(["date", "match_id", "team"]).reset_index(drop=True)
    groups = []
    for _, g in df.groupby("match_id", sort=False):
        g = g.reset_index(drop=True)
        if len(g) == 1:
            g = pd.concat([g, pd.DataFrame([make_reverse_row(g.iloc[0])])], ignore_index=True)
        groups.append(g)
    return groups

def build_actual_train_features(train_df: pd.DataFrame) -> pd.DataFrame:
    team_states = defaultdict(new_team_state)
    h2h_states = defaultdict(new_h2h_state)
    feature_rows = []
    for g in prepare_match_groups(train_df):
        for _, row in g.iterrows():
            feats = build_feature_row(
                row,
                team_states[str(row["team"])],
                team_states[str(row["opponent"])],
                h2h_states[(str(row["team"]), str(row["opponent"]))],
            )
            feats["team_goals"] = int(row["team_goals"])
            feats["opp_goals"] = int(row["opp_goals"])
            feats["total_goals"] = int(row["team_goals"] + row["opp_goals"])
            feats["goal_diff"] = int(row["team_goals"] - row["opp_goals"])
            feature_rows.append(feats)
        update_match_states(g.iloc[0], g.iloc[1], int(g.iloc[0]["team_goals"]), int(g.iloc[0]["opp_goals"]), team_states, h2h_states)
    return pd.DataFrame(feature_rows).sort_values(["date", "match_id", "team"]).reset_index(drop=True)

train_feat = build_actual_train_features(train)
train_feat.head(3)


,Id,match_id,date,team,opponent,gender,tournament,venue_country,confederation_team,confederation_opp,is_home,neutral,tournament_weight,year,month,dayofyear,dayofweek,month_sin,month_cos,doy_sin,doy_cos,same_conf,team_matches_played,opp_matches_played,rest_days_team,rest_days_opp,team_points_last5,opp_points_last5,team_points_last10,opp_points_last10,team_gd_last5,opp_gd_last5,team_avg_goals_last5,opp_avg_goals_last5,team_avg_conceded_last5,opp_avg_conceded_last5,team_avg_goals_last10,opp_avg_goals_last10,team_avg_conceded_last10,opp_avg_conceded_last10,team_win_rate_last10,opp_win_rate_last10,team_draw_rate_last10,opp_draw_rate_last10,team_home_points_last5,opp_away_points_last5,team_home_gf_last5,opp_away_gf_last5,elo_team,elo_opp,elo_diff,h2h_points_last5,h2h_gd_last5,h2h_avg_goals_last5,h2h_avg_conceded_last5,h2h_matches,population_team_log,population_opp_log,gdp_team_log,gdp_opp_log,altitude_venue_clean,distance_travel_team_clean,distance_travel_opp_clean,temperature_venue_clean,matches_played_diff,rest_days_diff,points_last5_diff,points_last10_diff,gd_last5_diff,avg_goals_last5_diff,avg_conceded_last5_diff,win_rate_last10_diff,draw_rate_last10_diff,pop_log_diff,gdp_log_diff,travel_diff,altitude_missing,temp_missing,team_goals,opp_goals,total_goals,goal_diff
0,M000001_England,M000001,1872-11-30,England,Scotland,M,Friendly,Scotland,UEFA,UEFA,0,0,0.96,1872,11,335,5,-0.5,8.660254e-01,-0.507415,0.861702,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1500.000000,1500.000000,0.00000,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,1,0,0,0,0
1,M000001_Scotland,M000001,1872-11-30,Scotland,England,M,Friendly,Scotland,UEFA,UEFA,1,0,0.96,1872,11,335,5,-0.5,8.660254e-01,-0.507415,0.861702,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1500.000000,1500.000000,0.00000,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,1,0,0,0,0
2,M000002_England,M000002,1873-03-08,England,Scotland,M,Friendly,England,UEFA,UEFA,1,0,0.96,1873,3,67,5,1.0,6.123234e-17,0.912846,0.408304,1,1,1,98.0,98.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,NaN,NaN,NaN,NaN,1501.569935,1498.430065,3.13987,1.0,0.0,0.0,0.0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,1,4,2,6,2


## Feature space, folds, dan model family


In [22]:
CAT_COLS = ["gender", "team", "opponent", "tournament", "venue_country", "confederation_team", "confederation_opp"]
DROP_COLS = {"Id", "match_id", "date", "team_goals", "opp_goals", "total_goals", "goal_diff"}
FEATURE_COLS = [c for c in train_feat.columns if c not in DROP_COLS]

def build_recent_time_folds(df: pd.DataFrame, n_folds: int = 5, start_quantile: float = 0.70):
    uniq_dates = np.array(sorted(pd.to_datetime(df["date"]).dt.normalize().unique()))
    start_idx = int(len(uniq_dates) * start_quantile)
    recent_dates = uniq_dates[start_idx:]
    blocks = np.array_split(recent_dates, n_folds)

    folds = []
    for i, block in enumerate(blocks, 1):
        if len(block) == 0:
            continue
        val_start = pd.Timestamp(block[0])
        val_end = pd.Timestamp(block[-1])
        tr_mask = df["date"] < val_start
        va_mask = (df["date"] >= val_start) & (df["date"] <= val_end)
        if int(tr_mask.sum()) == 0 or int(va_mask.sum()) == 0:
            continue
        folds.append({
            "fold": i,
            "val_start": val_start,
            "val_end": val_end,
            "train_mask": tr_mask.values,
            "valid_mask": va_mask.values,
        })
    return folds

FOLDS = build_recent_time_folds(train_feat, n_folds=N_FOLDS, start_quantile=FOLD_START_QUANTILE)
pd.DataFrame([{
    "fold": f["fold"],
    "val_start": f["val_start"].date(),
    "val_end": f["val_end"].date(),
    "train_rows": int(f["train_mask"].sum()),
    "valid_rows": int(f["valid_mask"].sum()),
} for f in FOLDS])


,fold,val_start,val_end,train_rows,valid_rows
0,1,1994-03-31,1997-09-26,40696,6218
1,2,1997-09-27,2001-01-24,46914,6794
2,3,2001-01-25,2004-05-16,53708,7462
3,4,2004-05-17,2007-12-17,61170,8644
4,5,2007-12-18,2011-08-04,69814,8958


In [23]:
def make_direct_model():
    return CatBoostRegressor(
        loss_function="Poisson",
        eval_metric="MAE",
        iterations=DIRECT_ITERS,
        depth=8,
        learning_rate=0.04,
        l2_leaf_reg=6.0,
        subsample=0.85,
        random_seed=SEED,
        verbose=False,
    )

def make_struct_model():
    return CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="MAE",
        iterations=STRUCT_ITERS,
        depth=8,
        learning_rate=0.04,
        l2_leaf_reg=6.0,
        subsample=0.85,
        random_seed=SEED,
        verbose=False,
    )

def finalize_feature_frame(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in CAT_COLS:
        out[c] = out[c].astype(str).fillna("__NA__")
    return out

def fit_model_family(train_frame: pd.DataFrame):
    X = finalize_feature_frame(train_frame[FEATURE_COLS])
    cat_idx = [X.columns.get_loc(c) for c in CAT_COLS]
    w = train_frame["tournament_weight"].values.astype(float)

    m_team = make_direct_model()
    m_opp = make_direct_model()
    m_total = make_struct_model()
    m_diff = make_struct_model()

    m_team.fit(X, train_frame["team_goals"], cat_features=cat_idx, sample_weight=w)
    m_opp.fit(X, train_frame["opp_goals"], cat_features=cat_idx, sample_weight=w)
    m_total.fit(X, train_frame["total_goals"], cat_features=cat_idx, sample_weight=w)
    m_diff.fit(X, train_frame["goal_diff"], cat_features=cat_idx, sample_weight=w)

    return {"direct_team": m_team, "direct_opp": m_opp, "struct_total": m_total, "struct_diff": m_diff}

def predict_raw_family(models: dict, feat_frame: pd.DataFrame) -> pd.DataFrame:
    X = finalize_feature_frame(feat_frame[FEATURE_COLS])
    d_team = np.clip(models["direct_team"].predict(X), 0, None)
    d_opp = np.clip(models["direct_opp"].predict(X), 0, None)
    s_total = np.clip(models["struct_total"].predict(X), 0, None)
    s_diff = models["struct_diff"].predict(X)
    s_team = np.clip((s_total + s_diff) / 2.0, 0, None)
    s_opp = np.clip((s_total - s_diff) / 2.0, 0, None)
    return pd.DataFrame({
        "direct_team": d_team, "direct_opp": d_opp,
        "struct_team": s_team, "struct_opp": s_opp,
        "struct_total": s_total, "struct_diff": s_diff,
    }, index=feat_frame.index)


## Blend, post-processing, dan sequential prediction


In [24]:
PROVISIONAL_BLEND = {"direct_weight": 0.70, "shrink": 0.95, "draw_pull": 0.08}
BLEND_GRID = {
    "direct_weight": np.round(np.linspace(0.45, 0.90, 10), 2),
    "shrink": [0.88, 0.92, 0.95, 0.98, 1.00],
    "draw_pull": [0.00, 0.04, 0.08, 0.12, 0.16, 0.20],
}

COMMON_SCORELINES = [
    (0,0), (1,0), (0,1), (1,1),
    (2,0), (0,2), (2,1), (1,2), (2,2),
    (3,0), (0,3), (3,1), (1,3), (3,2), (2,3), (3,3),
    (4,0), (0,4), (4,1), (1,4), (4,2), (2,4), (4,3), (3,4),
]

def blend_continuous(direct_team, direct_opp, struct_team, struct_opp, direct_weight=0.70, shrink=0.95, draw_pull=0.08):
    pred_team = direct_weight * direct_team + (1.0 - direct_weight) * struct_team
    pred_opp = direct_weight * direct_opp + (1.0 - direct_weight) * struct_opp
    total = (pred_team + pred_opp) * shrink
    diff = (pred_team - pred_opp) * (1.0 - draw_pull)
    out_team = np.clip((total + diff) / 2.0, 0, None)
    out_opp = np.clip((total - diff) / 2.0, 0, None)
    return out_team, out_opp

def choose_integer_score(
    pred_team: float,
    pred_opp: float,
    max_goals: int = MAX_GOALS,
    radius_true: int = 2,
    sigma: float = 0.85,
) -> Tuple[int, int]:
    """
    Memilih skor integer terbaik dengan pendekatan expected-loss:
    - continuous prediction dianggap pusat belief
    - kita bentuk beberapa plausible true score di sekitar prediksi continuous
    - untuk setiap kandidat skor integer, hitung expected official loss
    - pilih kandidat dengan expected loss terkecil

    Ini lebih selaras dengan metric baru dibanding round() biasa.
    """

    # Kandidat skor prediksi
    bt = int(np.floor(pred_team))
    bo = int(np.floor(pred_opp))

    candidate_scores = set(COMMON_SCORELINES)
    for t in range(max(0, bt - 2), min(max_goals, bt + 3) + 1):
        for o in range(max(0, bo - 2), min(max_goals, bo + 3) + 1):
            candidate_scores.add((t, o))

    plausible_truth = []
    t_min = max(0, int(np.floor(pred_team)) - radius_true)
    t_max = min(max_goals, int(np.ceil(pred_team)) + radius_true)
    o_min = max(0, int(np.floor(pred_opp)) - radius_true)
    o_max = min(max_goals, int(np.ceil(pred_opp)) + radius_true)

    for tt in range(t_min, t_max + 1):
        for oo in range(o_min, o_max + 1):
            plausible_truth.append((tt, oo))

    if len(plausible_truth) == 0:
        plausible_truth = [(int(round(pred_team)), int(round(pred_opp)))]

    truth_weights = []
    for tt, oo in plausible_truth:
        dist2 = (tt - pred_team) ** 2 + (oo - pred_opp) ** 2
        w = np.exp(-dist2 / (2 * sigma ** 2))
        truth_weights.append(w)

    truth_weights = np.asarray(truth_weights, dtype=float)
    truth_weights = truth_weights / truth_weights.sum()

    best_pair = None
    best_loss = np.inf

    for cand_t, cand_o in candidate_scores:
        exp_loss = 0.0

        for (tt, oo), w in zip(plausible_truth, truth_weights):
            loss = official_match_loss(
                y_team_true=[tt],
                y_opp_true=[oo],
                y_team_pred=[cand_t],
                y_opp_pred=[cand_o],
            )[0]
            exp_loss += w * loss

        exp_loss += 1e-4 * (
            abs(cand_t - pred_team) + abs(cand_o - pred_opp)
        )

        if exp_loss < best_loss:
            best_loss = exp_loss
            best_pair = (cand_t, cand_o)

    return best_pair

def replay_history(history_df: pd.DataFrame):
    team_states = defaultdict(new_team_state)
    h2h_states = defaultdict(new_h2h_state)
    for g in prepare_match_groups(history_df):
        update_match_states(g.iloc[0], g.iloc[1], int(g.iloc[0]["team_goals"]), int(g.iloc[0]["opp_goals"]), team_states, h2h_states)
    return team_states, h2h_states

def consolidate_pair_predictions(raw_pair: pd.DataFrame) -> Tuple[float, float]:
    if len(raw_pair) == 1:
        return float(raw_pair.iloc[0]["pred_team_cont"]), float(raw_pair.iloc[0]["pred_opp_cont"])
    row0, row1 = raw_pair.iloc[0], raw_pair.iloc[1]
    team0_goals = np.nanmean([row0["pred_team_cont"], row1["pred_opp_cont"]])
    opp0_goals = np.nanmean([row0["pred_opp_cont"], row1["pred_team_cont"]])
    return float(team0_goals), float(opp0_goals)

def sequential_predict_block(block_df: pd.DataFrame, models: dict, team_states: dict, h2h_states: dict, blend_params: Optional[dict] = None, use_actual_for_update: bool = False):
    if blend_params is None:
        blend_params = PROVISIONAL_BLEND.copy()

    rows_out = []
    for g in prepare_match_groups(block_df):
        feat_rows = []
        for _, row in g.iterrows():
            feats = build_feature_row(
                row,
                team_states[str(row["team"])],
                team_states[str(row["opponent"])],
                h2h_states[(str(row["team"]), str(row["opponent"]))],
            )
            feat_rows.append(feats)

        feat_frame = pd.DataFrame(feat_rows)
        raw_preds = predict_raw_family(models, feat_frame)
        feat_frame = pd.concat([feat_frame.reset_index(drop=True), raw_preds.reset_index(drop=True)], axis=1)

        pred_team_cont, pred_opp_cont = blend_continuous(
            feat_frame["direct_team"].values,
            feat_frame["direct_opp"].values,
            feat_frame["struct_team"].values,
            feat_frame["struct_opp"].values,
            direct_weight=blend_params["direct_weight"],
            shrink=blend_params["shrink"],
            draw_pull=blend_params["draw_pull"],
        )

        feat_frame["pred_team_cont"] = pred_team_cont
        feat_frame["pred_opp_cont"] = pred_opp_cont
        match_team_cont, match_opp_cont = consolidate_pair_predictions(feat_frame[["pred_team_cont", "pred_opp_cont"]])
        match_team_int, match_opp_int = choose_integer_score(match_team_cont, match_opp_cont)

        for i, row in feat_frame.iterrows():
            if i == 0:
                final_team, final_opp = match_team_int, match_opp_int
                cont_team, cont_opp = match_team_cont, match_opp_cont
            else:
                final_team, final_opp = match_opp_int, match_team_int
                cont_team, cont_opp = match_opp_cont, match_team_cont

            row_dict = row.to_dict()
            row_dict["pred_team_final"] = final_team
            row_dict["pred_opp_final"] = final_opp
            row_dict["pair_team_cont"] = cont_team
            row_dict["pair_opp_cont"] = cont_opp
            if "team_goals" in g.columns and pd.notna(g.iloc[min(i, len(g)-1)].get("team_goals", np.nan)):
                row_dict["team_goals"] = int(g.iloc[min(i, len(g)-1)]["team_goals"])
                row_dict["opp_goals"] = int(g.iloc[min(i, len(g)-1)]["opp_goals"])
            rows_out.append(row_dict)

        if use_actual_for_update and "team_goals" in g.iloc[0].index and pd.notna(g.iloc[0]["team_goals"]):
            goals_a = int(g.iloc[0]["team_goals"]); goals_b = int(g.iloc[0]["opp_goals"])
        else:
            goals_a = int(match_team_int); goals_b = int(match_opp_int)
        update_match_states(g.iloc[0], g.iloc[1], goals_a, goals_b, team_states, h2h_states)

    out_df = pd.DataFrame(rows_out).sort_values(["date", "match_id", "team"]).reset_index(drop=True)
    return out_df, team_states, h2h_states


## Walk-forward OOF, tuning blend, dan prediksi final


In [25]:
oof_frames = []
fold_scores = []

for fold_info in FOLDS:
    train_block = train_feat.loc[fold_info["train_mask"]].copy()
    valid_raw = train.loc[fold_info["valid_mask"]].copy()

    models = fit_model_family(train_block)
    team_states, h2h_states = replay_history(train[train["date"] < fold_info["val_start"]].copy())

    valid_pred_df, _, _ = sequential_predict_block(
        block_df=valid_raw,
        models=models,
        team_states=team_states,
        h2h_states=h2h_states,
        blend_params=PROVISIONAL_BLEND,
        use_actual_for_update=False,
    )

    valid_truth = train.loc[fold_info["valid_mask"], ["Id", "team_goals", "opp_goals", "tournament"]].copy()
    valid_pred_df = valid_pred_df.merge(valid_truth, on="Id", how="left", suffixes=("", "_true_fix"))
    if "team_goals_true_fix" in valid_pred_df.columns:
        valid_pred_df["team_goals"] = valid_pred_df["team_goals_true_fix"]
        valid_pred_df["opp_goals"] = valid_pred_df["opp_goals_true_fix"]
        valid_pred_df = valid_pred_df.drop(columns=["team_goals_true_fix", "opp_goals_true_fix", "tournament_true_fix"], errors="ignore")

    fold_aw = awmae_score(
        valid_pred_df["team_goals"].values,
        valid_pred_df["opp_goals"].values,
        valid_pred_df["pred_team_final"].values,
        valid_pred_df["pred_opp_final"].values,
        valid_pred_df["tournament"].apply(get_tournament_weight).values,
    )

    fold_scores.append({"fold": fold_info["fold"], "awmae": fold_aw})
    oof_frames.append(valid_pred_df)

oof_df = pd.concat(oof_frames, ignore_index=True)
pd.DataFrame(fold_scores)

,fold,awmae
0,1,3.067992
1,2,3.106452
2,3,3.249429
3,4,3.011160
4,5,2.959965


In [ ]:
def evaluate_blend_grid(oof_df: pd.DataFrame, grid: dict) -> pd.DataFrame:
    rows = []
    weights = oof_df["tournament"].apply(get_tournament_weight).values

    for dw in grid["direct_weight"]:
        for shrink in grid["shrink"]:
            for draw_pull in grid["draw_pull"]:
                pred_team_cont, pred_opp_cont = blend_continuous(
                    oof_df["direct_team"].values,
                    oof_df["direct_opp"].values,
                    oof_df["struct_team"].values,
                    oof_df["struct_opp"].values,
                    direct_weight=float(dw),
                    shrink=float(shrink),
                    draw_pull=float(draw_pull),
                )
                pred_team_int, pred_opp_int = [], []
                for pt, po in zip(pred_team_cont, pred_opp_cont):
                    a, b = choose_integer_score(pt, po)
                    pred_team_int.append(a); pred_opp_int.append(b)

                score = awmae_score(
                    oof_df["team_goals"].values, oof_df["opp_goals"].values,
                    np.array(pred_team_int), np.array(pred_opp_int), weights,
                )
                rows.append({"direct_weight": float(dw), "shrink": float(shrink), "draw_pull": float(draw_pull), "awmae": score})
    return pd.DataFrame(rows).sort_values("awmae").reset_index(drop=True)

blend_search = evaluate_blend_grid(oof_df, BLEND_GRID)
BEST_BLEND = blend_search.iloc[0].to_dict()
blend_search.head(10)

In [ ]:
final_models = fit_model_family(train_feat)
team_states_full, h2h_states_full = replay_history(train.copy())

test_pred_df, _, _ = sequential_predict_block(
    block_df=test.copy(),
    models=final_models,
    team_states=team_states_full,
    h2h_states=h2h_states_full,
    blend_params={
        "direct_weight": float(BEST_BLEND["direct_weight"]),
        "shrink": float(BEST_BLEND["shrink"]),
        "draw_pull": float(BEST_BLEND["draw_pull"]),
    },
    use_actual_for_update=False,
)

submission = pd.DataFrame({
    "Id": test_pred_df["Id"],
    "team_goals": test_pred_df["pred_team_final"].astype(int).clip(lower=0),
    "opp_goals": test_pred_df["pred_opp_final"].astype(int).clip(lower=0),
}).sort_values("Id").reset_index(drop=True)

submission.to_csv("submission_awmae_rebuild.csv", index=False)
print("Saved: submission_awmae_rebuild.csv")
display(submission.head())


Saved: submission_awmae_rebuild.csv


,Id,team_goals,opp_goals
0,M034984_Mauritius,1,1
1,M034984_Seychelles,1,1
2,M034985_Comoros,1,1
3,M034985_Maldives,1,1
4,M034986_Madagascar,1,1
